In [ ]:
# fx_monthly.csv was already made by Kayode (data/processed/fx_monthly.csv).
import glob
import pandas as pd

rows = []
files = sorted(glob.glob('../data/raw/pms_reports/*.xlsx'))

# 1. Loop through raw Excel reports
for f in files:
    if '~$' in f:  # skip temporary files created while a workbook is open in Excel
        continue

    sheets = pd.read_excel(f, sheet_name=None, header=None)  # every sheet in the file

    # 2. Find the sheet containing the 'State' table
    ws = None
    for sh in sheets.values():
        if (sh[0].astype(str).str.strip() == 'State').any():
            ws = sh
            break

    if ws is None:
        print('No State table found in', f)
        continue

    # 3. Extract header dates and the national AVERAGE row
    hdr = ws.index[ws[0].astype(str).str.strip() == 'State'][0]
    dates = pd.to_datetime(ws.loc[hdr, [1, 2, 3]]).tolist()  # same month last year, last month, this month
    avg = ws[ws[0].astype(str).str.strip().str.upper() == 'AVERAGE'].iloc[0]
    report_month = dates[2]                                   # the month this report is for

    # 4. Append each month's price record with its source report month
    for col, dt in zip([1, 2, 3], dates):
        rows.append({
            'month': dt.to_period('M').to_timestamp(),
            'pms_price': float(avg[col]),
            'report_month': report_month,
        })

# 5. Deduplicate using the newest report for each month and sort chronologically
pms = (
    pd.DataFrame(rows)
    .sort_values('report_month')  # oldest report first, so keep='last' keeps the newest
    .drop_duplicates('month', keep='last')
    .sort_values('month')
    [['month', 'pms_price']]
)

# 6. Check coverage (January 2023 to May 2026) and export
expected = pd.date_range('2023-01-01', '2026-05-01', freq='MS')
missing = [d.strftime('%Y-%m') for d in expected if d not in set(pms['month'])]
print('Rows:', len(pms), '| missing months:', missing)
print(pms[pms['month'] >= '2023-01-01'])  # spot check: Oct 2023 is about 630.6 and Oct 2024 about 1184.8

pms.to_csv('../data/processed/pms_monthly.csv', index=False)